# Vector Symbolic Architectures (VSA) with Grilly

**Compositional reasoning with high-dimensional distributed representations.**

Vector Symbolic Architectures (also called Hyperdimensional Computing) represent
symbols as high-dimensional vectors and use algebraic operations to compose,
store, and retrieve structured information.

This notebook covers:
1. GPU/CPU detection
2. Core VSA concepts: bind, bundle, similarity
3. Creating random hypervectors with `BlockCodeOps`
4. Binding and similarity checks
5. Key-value memory: bind(key, value), retrieve via unbind
6. Cleanup/purification with codebook lookup
7. Capacity test: how many items can a bundle hold?

Grilly provides VSA operations in `grilly.experimental.vsa`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import grilly
from grilly.experimental.vsa import BlockCodeOps

# --- Backend detection ---
try:
    from grilly._bridge import is_vulkan_available
    DEVICE = "vulkan" if is_vulkan_available() else "cpu"
except (ImportError, AttributeError):
    DEVICE = "cpu"

print(f"grilly {grilly.__version__} | backend: {DEVICE}")
np.random.seed(42)

## 1. What Are Vector Symbolic Architectures?

In VSA, every concept (word, role, relation, object) is represented by a
high-dimensional vector called a **hypervector**. Three core operations let
you build structured representations:

| Operation | Symbol | What it does | Key property |
|-----------|--------|-------------|---------------|
| **Bind** | A * B | Associates two concepts | Dissimilar to both A and B |
| **Bundle** | A + B | Superposes concepts | Similar to both A and B |
| **Similarity** | sim(A, B) | Measures relatedness | Cosine similarity |

### Block Codes

Grilly uses **block codes** -- vectors of shape `(k, l)` where `k` is the
number of blocks and `l` is the block length. Binding uses per-block circular
convolution. We use small dimensions here (`k=8, l=64`, total D=512) to
keep things fast.

In [ ]:
# Initialize block code operations with small dimensions
K_BLOCKS = 8    # Number of blocks
L_BLOCK = 64    # Block length
D_VSA = K_BLOCKS * L_BLOCK  # Total dimensionality = 512

ops = BlockCodeOps(k=K_BLOCKS, l=L_BLOCK)

print(f"VSA Configuration:")
print(f"  k (blocks)     : {K_BLOCKS}")
print(f"  l (block length): {L_BLOCK}")
print(f"  D (total dims) : {D_VSA}")

## 2. Creating Random Hypervectors

Each concept gets a random hypervector. In high dimensions, random vectors
are nearly orthogonal to each other -- this is the blessing of dimensionality
that makes VSA work.

In [ ]:
# Create random hypervectors for several concepts
apple = ops.random_vector()
banana = ops.random_vector()
cherry = ops.random_vector()
red = ops.random_vector()
yellow = ops.random_vector()
sweet = ops.random_vector()

print(f"Vector shape: {apple.shape}  (k={K_BLOCKS} blocks x l={L_BLOCK})")
print(f"Vector dtype: {apple.dtype}")
print()

# Verify near-orthogonality of random vectors
pairs = [
    ('apple', 'apple', apple, apple),
    ('apple', 'banana', apple, banana),
    ('apple', 'cherry', apple, cherry),
    ('red', 'yellow', red, yellow),
]

print("Similarity between random hypervectors:")
print("-" * 45)
for name_a, name_b, vec_a, vec_b in pairs:
    sim = ops.similarity(vec_a, vec_b)
    bar = '#' * int(abs(sim) * 40)
    print(f"  sim({name_a:6s}, {name_b:6s}) = {sim:+.4f}  {bar}")

print("\nRandom vectors are nearly orthogonal (similarity close to 0).")
print("Self-similarity is 1.0 (identical vectors).")

## 3. Binding: Associating Two Concepts

The **bind** operation creates a new vector representing the association of
two concepts. Crucially, the result is dissimilar to both inputs -- you
cannot tell which concepts were bound just by looking at the result.

Binding is reversible: `unbind(bind(A, B), A) = B`.

In [ ]:
# Bind apple with red: "apple is red"
apple_is_red = ops.bind(apple, red)

print("Binding: apple * red = apple_is_red")
print()
print("Similarity checks:")
print(f"  sim(apple_is_red, apple)  = {ops.similarity(apple_is_red, apple):+.4f}  (dissimilar!)")
print(f"  sim(apple_is_red, red)    = {ops.similarity(apple_is_red, red):+.4f}  (dissimilar!)")
print(f"  sim(apple_is_red, banana) = {ops.similarity(apple_is_red, banana):+.4f}  (also dissimilar)")

print()
print("Unbinding (retrieving the other element):")

# Unbind with apple to recover red
recovered_red = ops.unbind(apple_is_red, apple)
print(f"  unbind(apple_is_red, apple):")
print(f"    sim(result, red)    = {ops.similarity(recovered_red, red):+.4f}  (recovered!)")
print(f"    sim(result, yellow) = {ops.similarity(recovered_red, yellow):+.4f}  (not yellow)")

# Unbind with red to recover apple
recovered_apple = ops.unbind(apple_is_red, red)
print(f"  unbind(apple_is_red, red):")
print(f"    sim(result, apple)  = {ops.similarity(recovered_apple, apple):+.4f}  (recovered!)")
print(f"    sim(result, banana) = {ops.similarity(recovered_apple, banana):+.4f}  (not banana)")

## 4. Bundling: Superposing Multiple Concepts

The **bundle** operation (element-wise addition) creates a vector that is
similar to all its components. This is how you represent a set or a
multi-attribute object.

In [ ]:
# Bundle: "fruit salad" = apple + banana + cherry
fruit_salad = ops.bundle([apple, banana, cherry])

print("Bundle: apple + banana + cherry = fruit_salad")
print()
print("Similarity to each component:")
print(f"  sim(fruit_salad, apple)  = {ops.similarity(fruit_salad, apple):+.4f}  (similar!)")
print(f"  sim(fruit_salad, banana) = {ops.similarity(fruit_salad, banana):+.4f}  (similar!)")
print(f"  sim(fruit_salad, cherry) = {ops.similarity(fruit_salad, cherry):+.4f}  (similar!)")
print(f"  sim(fruit_salad, red)    = {ops.similarity(fruit_salad, red):+.4f}  (not in bundle)")
print(f"  sim(fruit_salad, sweet)  = {ops.similarity(fruit_salad, sweet):+.4f}  (not in bundle)")

print("\nThe bundle is similar to its components but not to unrelated concepts.")

## 5. Key-Value Memory with Bind + Bundle

By combining bind and bundle, we can build an associative memory:

1. **Store**: `memory = bind(key1, val1) + bind(key2, val2) + ...`
2. **Retrieve**: `unbind(memory, key1)` recovers something similar to `val1`

This is analogous to a dictionary/hash map, but implemented purely with
vector algebra.

In [ ]:
# Define role vectors (keys)
ROLE_color = ops.random_vector()
ROLE_taste = ops.random_vector()
ROLE_name = ops.random_vector()

# Store: "apple" record = {color: red, taste: sweet, name: apple}
apple_record = ops.bundle([
    ops.bind(ROLE_color, red),
    ops.bind(ROLE_taste, sweet),
    ops.bind(ROLE_name, apple),
])

print("Stored record: {color: red, taste: sweet, name: apple}")
print()

# Retrieve: "What color is the apple?"
query_result = ops.unbind(apple_record, ROLE_color)

# Check similarity against all known values
codebook = {
    'red': red, 'yellow': yellow, 'sweet': sweet,
    'apple': apple, 'banana': banana, 'cherry': cherry,
}

print("Query: 'What color is the apple?'")
print("  unbind(apple_record, ROLE_color) similarities:")
sims = {}
for name, vec in codebook.items():
    sim = ops.similarity(query_result, vec)
    sims[name] = sim
    marker = " <-- MATCH" if sim > 0.2 else ""
    print(f"    {name:8s}: {sim:+.4f}{marker}")

print()
print("Query: 'What taste does the apple have?'")
taste_result = ops.unbind(apple_record, ROLE_taste)
for name, vec in codebook.items():
    sim = ops.similarity(taste_result, vec)
    marker = " <-- MATCH" if sim > 0.2 else ""
    print(f"    {name:8s}: {sim:+.4f}{marker}")

## 6. Cleanup / Purification

Retrieved vectors are noisy (approximate). **Cleanup** finds the closest
match in a codebook of known vectors. This is a simple argmax over
similarities.

In [ ]:
def cleanup(noisy_vector, codebook, ops):
    """Find the closest clean vector in the codebook.

    Args:
        noisy_vector: The approximate/noisy vector to clean up
        codebook: Dict of {name: clean_vector}
        ops: BlockCodeOps instance

    Returns:
        (best_name, best_similarity, best_vector)
    """
    best_name = None
    best_sim = -np.inf
    best_vec = None

    for name, vec in codebook.items():
        sim = ops.similarity(noisy_vector, vec)
        if sim > best_sim:
            best_sim = sim
            best_name = name
            best_vec = vec

    return best_name, best_sim, best_vec

# Demonstrate cleanup on the noisy retrieval
query_result = ops.unbind(apple_record, ROLE_color)
name, sim, clean_vec = cleanup(query_result, codebook, ops)

print(f"Noisy retrieval: unbind(apple_record, ROLE_color)")
print(f"Cleanup result : '{name}' (similarity = {sim:.4f})")
print(f"Correct?       : {'Yes' if name == 'red' else 'No'}")
print()

# After cleanup, the vector is exact (no noise)
print(f"sim(clean, red) = {ops.similarity(clean_vec, red):+.4f}  (exact match)")

## 7. Capacity Test: How Many Items Can a Bundle Hold?

A fundamental question in VSA: how many items can you bundle together
before retrieval accuracy degrades? Let's test this experimentally.

We create N random key-value pairs, bundle them into a single memory
vector, then try to retrieve each value and check if cleanup finds
the correct answer.

In [ ]:
def test_capacity(ops, n_items, n_trials=5):
    """Test bundle capacity: store n_items key-value pairs, measure retrieval accuracy."""
    accuracies = []

    for trial in range(n_trials):
        # Create random keys and values
        keys = [ops.random_vector() for _ in range(n_items)]
        values = [ops.random_vector() for _ in range(n_items)]

        # Build memory: sum of bind(key_i, value_i)
        bound_pairs = [ops.bind(k, v) for k, v in zip(keys, values)]
        memory = ops.bundle(bound_pairs)

        # Retrieve each value and check with cleanup
        correct = 0
        value_codebook = {i: v for i, v in enumerate(values)}

        for i in range(n_items):
            retrieved = ops.unbind(memory, keys[i])
            # Find best match among all values
            best_idx = max(value_codebook.keys(),
                          key=lambda j: ops.similarity(retrieved, values[j]))
            if best_idx == i:
                correct += 1

        accuracies.append(correct / n_items)

    return np.mean(accuracies), np.std(accuracies)

# Test different numbers of items
item_counts = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50]
means = []
stds = []

print(f"Capacity test: k={K_BLOCKS}, l={L_BLOCK}, D={D_VSA}")
print("-" * 50)

for n in item_counts:
    mean_acc, std_acc = test_capacity(ops, n, n_trials=5)
    means.append(mean_acc)
    stds.append(std_acc)
    bar = '#' * int(mean_acc * 30)
    print(f"  {n:3d} items: {mean_acc*100:5.1f}% +/- {std_acc*100:4.1f}%  {bar}")

print()
print("Tip: Increase k and l for higher capacity.")
print(f"Production CubeMind uses k=80, l=128, D=10240.")

In [ ]:
# Plot capacity curve
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

means_arr = np.array(means) * 100
stds_arr = np.array(stds) * 100

ax.plot(item_counts, means_arr, 'o-', color='#2c3e50', linewidth=2, markersize=6)
ax.fill_between(item_counts, means_arr - stds_arr, means_arr + stds_arr,
                alpha=0.2, color='#3498db')
ax.axhline(y=90, color='#e74c3c', linestyle='--', alpha=0.5, label='90% threshold')
ax.axhline(y=50, color='#95a5a6', linestyle=':', alpha=0.5, label='Chance (50%)')

ax.set_xlabel('Number of Items Bundled')
ax.set_ylabel('Retrieval Accuracy (%)')
ax.set_title(f'VSA Bundle Capacity (k={K_BLOCKS}, l={L_BLOCK}, D={D_VSA})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 8. Structured Representation: A Complete Example

Let's put it all together to represent and query a small knowledge base.
We represent two fruits with multiple attributes and store them in a
single bundle.

In [ ]:
# Define roles (keys)
ROLE_item = ops.random_vector()
ROLE_color = ops.random_vector()
ROLE_taste = ops.random_vector()
ROLE_size = ops.random_vector()

# Define fillers (values)
apple = ops.random_vector()
banana = ops.random_vector()
red = ops.random_vector()
yellow = ops.random_vector()
sweet = ops.random_vector()
tart = ops.random_vector()
small = ops.random_vector()
medium = ops.random_vector()

all_fillers = {
    'apple': apple, 'banana': banana,
    'red': red, 'yellow': yellow,
    'sweet': sweet, 'tart': tart,
    'small': small, 'medium': medium,
}

# Build records
apple_record = ops.bundle([
    ops.bind(ROLE_item, apple),
    ops.bind(ROLE_color, red),
    ops.bind(ROLE_taste, tart),
    ops.bind(ROLE_size, small),
])

banana_record = ops.bundle([
    ops.bind(ROLE_item, banana),
    ops.bind(ROLE_color, yellow),
    ops.bind(ROLE_taste, sweet),
    ops.bind(ROLE_size, medium),
])

# Query each record
queries = [
    ("What color is the apple?", apple_record, ROLE_color, 'red'),
    ("What taste is the apple?", apple_record, ROLE_taste, 'tart'),
    ("What size is the apple?", apple_record, ROLE_size, 'small'),
    ("What color is the banana?", banana_record, ROLE_color, 'yellow'),
    ("What taste is the banana?", banana_record, ROLE_taste, 'sweet'),
    ("What size is the banana?", banana_record, ROLE_size, 'medium'),
]

print("Knowledge Base Queries")
print("=" * 60)
all_correct = True
for question, record, role, expected in queries:
    retrieved = ops.unbind(record, role)
    answer, sim, _ = cleanup(retrieved, all_fillers, ops)
    correct = answer == expected
    if not correct:
        all_correct = False
    status = "OK" if correct else "MISS"
    print(f"  Q: {question}")
    print(f"  A: {answer} (sim={sim:.4f}) [{status}]")
    print()

print(f"All queries correct: {all_correct}")

## Summary

In this notebook you learned:

- **Hypervectors**: high-dimensional random vectors that are nearly orthogonal
- **Bind (A * B)**: creates associations, dissimilar to both inputs, reversible
- **Bundle (A + B)**: superposes concepts, similar to all components
- **Similarity**: cosine similarity measures relatedness
- **Key-value memory**: bind(role, filler) + bundle for structured records
- **Cleanup**: argmax similarity lookup to recover clean symbols
- **Capacity**: limited by dimensionality -- D=512 handles ~5-10 items,
  production D=10240 handles many more

### Key Takeaways
- VSA enables **compositional reasoning** without gradient training
- Block codes (`k` blocks of length `l`) give efficient per-block operations
- The 3-level GPU fallback (C++/Vulkan -> Python GPU -> numpy) is automatic
- CubeMind uses VSA for zero-shot rule detection on I-RAVEN

### Next Steps
- **Notebook 05**: Attention mechanisms and transformers